# 05 - Publish a STAC catalog

[STAC](https://stacspec.org) (SpatioTemporal Asset Catalog) describes geospatial data with JSON documents:

* **Item**: a GeoJSON Feature, with the spatio-temporal footprint of one or more files (the **assets**);
* **Collection**: a group of items, with its own extent, license, description (and possibly its own assets);
* **Catalog**: the entry point, linking to collections and other catalogs.

**Extensions** add fields: here we use [projection](https://github.com/stac-extensions/projection) (CRS, shape, transform) and [datacube](https://github.com/stac-extensions/datacube) (dimensions and variables of the Zarr stores).

We build the catalog with [pystac](https://pystac.readthedocs.io), which takes care of the JSON structure, the extensions and the links between documents. The result is a *static* catalog: JSON files on the object store, which can be browsed without any server. At the end, we also export the items as [stac-geoparquet](https://github.com/stac-utils/stac-geoparquet) and search them with [rustac](https://stac-utils.github.io/rustac-py).

In [ ]:
import os
import pathlib
import warnings
from datetime import datetime

import fsspec
import obstore
import pystac
import rasterio
import rustac
import shapely
import xarray as xr

from pystac.extensions.datacube import DatacubeExtension, Dimension, Variable
from pystac.extensions.projection import ProjectionExtension
from pystac.utils import datetime_to_str
from rasterio.warp import transform_geom

from utils import ENDPOINT_URL, USER_PREFIX_URI, gdal_env, https_url

In [ ]:
fs = fsspec.filesystem("s3", endpoint_url=ENDPOINT_URL)

CATALOG_ROOT = f"{USER_PREFIX_URI}/stac"

## Items for the COGs

We read CRS, shape and bounds from the file headers (only a few kB per file!).

In [ ]:
def raster_item(href: str, start: str, end: str) -> pystac.Item:
    with gdal_env(), rasterio.open(href) as src:
        geometry = transform_geom(src.crs, "EPSG:4326", shapely.box(*src.bounds).__geo_interface__)
        item = pystac.Item(
            id=pathlib.PurePath(href).stem,
            geometry=geometry,
            bbox=list(shapely.geometry.shape(geometry).bounds),
            datetime=None,
            start_datetime=datetime.fromisoformat(start),
            end_datetime=datetime.fromisoformat(end),
            properties={},
        )
        ProjectionExtension.ext(item, add_if_missing=True).apply(
            code=src.crs.to_string(),
            shape=list(src.shape),
            transform=list(src.transform)[:6],
            bbox=list(src.bounds),
        )
    item.add_asset("data", pystac.Asset(https_url(href), media_type=pystac.MediaType.COG, roles=["data"]))
    return item

In [ ]:
dsm_items = [
    raster_item(f"s3://{p}", "2020-01-01T00:00:00Z", "2022-12-31T23:59:59Z")
    for p in fs.glob(f"{USER_PREFIX_URI}/AHN-4/DSM05/cog/*.tif")
]
dsm_items[0]

In [ ]:
rgb_items = [
    raster_item(f"s3://{p}", "2024-01-01T00:00:00Z", "2024-12-31T23:59:59Z")
    for p in fs.glob(f"{USER_PREFIX_URI}/aerial-images-NL-2024/RGB/cog/*.tif")
]
rgb_items[0]

## Collections

The extent of a collection is derived from its items (`pystac.Extent.from_items`). The temperature collection has no items: the Zarr stores are collection-level assets, described with the datacube extension.

In [ ]:
def collection(id: str, description: str, items: list[pystac.Item], license: str) -> pystac.Collection:
    coll = pystac.Collection(id, description, extent=pystac.Extent.from_items(items), license=license)
    coll.add_items(items)  # also sets the "collection" field of the items
    return coll

In [ ]:
dsm_collection = collection("ahn4-dsm05", "AHN-4 Digital Surface Model (0.5 m), as COGs", dsm_items, "CC0-1.0")
rgb_collection = collection("aerial-images-2024", "Aerial images 2024 (RGB, 8 cm), as COGs", rgb_items, "CC-BY-4.0")

In [ ]:
zarr_stores = {
    pathlib.PurePath(p).stem: f"s3://{p}"
    for p in fs.glob(f"{USER_PREFIX_URI}/KNMI-Tg1-5/zarr/*.zarr")
}
ds = xr.open_zarr(next(iter(zarr_stores.values())), consolidated=False, storage_options={"endpoint_url": ENDPOINT_URL})
start, end = ds.indexes["time"][[0, -1]].to_pydatetime()

temperature_collection = pystac.Collection(
    "knmi-tg1",
    "KNMI gridded daily mean temperature (Tg1, 1 km), 2025, as Zarr",
    extent=pystac.Extent(
        pystac.SpatialExtent([[float(ds.lon.min()), float(ds.lat.min()), float(ds.lon.max()), float(ds.lat.max())]]),
        pystac.TemporalExtent([[start, end]]),
    ),
    license="CC-BY-4.0",
)

for name, href in zarr_stores.items():
    temperature_collection.add_asset(
        f"zarr-{name}",
        pystac.Asset(
            https_url(href),
            media_type="application/vnd.zarr; version=3",
            roles=["data"],
            description=f"Zarr store, {name} chunk layout",
        ),
    )
temperature_collection.add_asset(
    "icechunk",
    pystac.Asset(
        https_url(f"{USER_PREFIX_URI}/KNMI-Tg1-5/icechunk"),
        media_type="application/vnd.zarr+icechunk",
        roles=["data", "references"],
        description="Icechunk repository with virtual references to the original NetCDF files",
    ),
)

DatacubeExtension.ext(temperature_collection, add_if_missing=True).apply(
    dimensions={
        "time": Dimension.from_dict({
            "type": "temporal", 
            "extent": [datetime_to_str(start), datetime_to_str(end)], 
            "step": "P1D"
        }),
        "x": Dimension.from_dict({
            "type": "spatial", 
            "axis": "x", 
            "extent": [float(ds.x.min()), float(ds.x.max())], 
            "step": 1000, 
            "reference_system": 28992
        }),
        "y": Dimension.from_dict({
            "type": "spatial", 
            "axis": "y", 
            "extent": [float(ds.y.min()), float(ds.y.max())], 
            "step": 1000, 
            "reference_system": 28992
        }),
    },
    variables={
        "prediction": Variable({
            "type": "data", 
            "dimensions": ["time", "y", "x"], 
            "unit": "degC", 
            "description": "Daily mean temperature"
        }),
    },
)
temperature_collection

## Link everything together and upload

`normalize_hrefs` sets the (public HTTPS) location of every document, following the standard STAC layout. We save a *self-contained* catalog, with relative links only, so the whole folder can be moved or copied anywhere: we write it locally and upload it to the object store.

In [ ]:
catalog = pystac.Catalog(
    "cng-forum-workshop",
    "Cloud-native versions of PDOK and KNMI datasets, created during the CNG Forum 2026 workshop",
)
catalog.add_children([dsm_collection, rgb_collection, temperature_collection])
catalog.normalize_hrefs(https_url(CATALOG_ROOT))
catalog.save(pystac.CatalogType.SELF_CONTAINED, dest_href="out/stac")

fs.put("out/stac/", CATALOG_ROOT, recursive=True)
root_href = catalog.get_self_href()
print(fs.tree(CATALOG_ROOT))

## Browse the catalog

The catalog is plain JSON on a publicly readable bucket with CORS enabled, so [STAC Browser](https://radiantearth.github.io/stac-browser) can load it directly. Click the link below:

In [ ]:
print(f"https://radiantearth.github.io/stac-browser/#/external/{root_href.removeprefix("https://")}")

Also check out the catalog on [stac-map](https://developmentseed.org/stac-map/):

In [ ]:
print(f"https://developmentseed.org/stac-map/?href={root_href}")

## stac-geoparquet

For large catalogs, crawling thousands of JSON files is slow. [stac-geoparquet](https://github.com/stac-utils/stac-geoparquet) stores all items in a single (cloud-optimized!) Parquet file, which can be queried with range requests, much like a COG. 

In [ ]:
def save_as_geoparquet(href: str, items: list[pystac.Item]) -> None:
    """Save a list of STAC items as GeoParquet."""
    prefix, filename = os.path.split(href)
    store = obstore.store.from_url(prefix)
    rustac.write_sync(
        href=filename, 
        value=[item.to_dict() for item in items], 
        store=store
    )

In [ ]:
save_as_geoparquet(f"{CATALOG_ROOT}/ahn4-dsm05.parquet", dsm_items)
save_as_geoparquet(f"{CATALOG_ROOT}/aerial-images-2024.parquet", rgb_items)

In [ ]:
# search directly on the remote parquet file (DuckDB under the hood)
res = rustac.search_sync(
    https_url(f"{CATALOG_ROOT}/aerial-images-2024.parquet"), 
    bbox=[5.10, 52.07, 5.11, 52.08]
)
len(res)